# Model Training

### Imports

In [ ]:
import os
os.environ.setdefault("MPLBACKEND", "Agg")

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import yaml
import numpy as np
from torch.utils.data import DataLoader
from pathlib import Path
from tqdm import tqdm
import sys
from IPython.display import display, clear_output
import ipywidgets as widgets



project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from pyfunctions.build_model import build_model_finetune
from pyfunctions.losses import SegmentationLoss, calculate_class_frequencies, calculate_class_weights_from_frequencies
from pyfunctions.metrics import calculate_metrics
from pyfunctions.dataload import create_datasets, create_dataloaders
from pyfunctions.warmup_scheduler import GradualWarmupScheduler

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False

print("All imports successful")

### Define Train Functions

In [ ]:
class SegmentationTrainer:
    """Trainer class for fine-tuning segmentation models"""
    
    def __init__(self, config):
        """Initialize the trainer with config"""
        self.config = config
        self.num_classes = None
        self.class_names = None
        
        # Jupyter notebook detection
        self.is_notebook = self._is_notebook()

        if torch.cuda.is_available():
            gpu_ids_config = config['hardware']['gpu']
            if isinstance(gpu_ids_config, (int, str)):
                gpu_id = str(gpu_ids_config).split(',')[0]
            elif isinstance(gpu_ids_config, list) and gpu_ids_config:
                gpu_id = str(gpu_ids_config[0]).split(',')[0]
            else:
                print("Warning: GPU configuration invalid or empty. Defaulting to cuda:0.")
                gpu_id = '0'
            self.device = torch.device(f'cuda:{gpu_id}')
            print(f"Using GPU: {gpu_id}")
        else:
            self.device = torch.device('cpu')
            print("Using CPU")


        self.checkpoint_dir = project_root / config['training']['checkpoint_dir']
        self.model_dir = project_root / config['training']['model_dir']

        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.model_dir.mkdir(parents=True, exist_ok=True)

        self.is_main_process = True
        self._setup_data()
        self.setup_model()
        self.setup_training()

        if self.config.get('wandb', {}).get('enable', False) and WANDB_AVAILABLE and self.is_main_process:
            wandb.init(
                project=self.config['wandb'].get('project', 'segmentation-finetuning'),
                name=self.config['wandb'].get('run_name', self.config.get('name', 'default-run')),
                config=self.config,
                dir=str(project_root)
            )

    def _is_notebook(self):
        """Check if running in Jupyter notebook"""
        try:
            from IPython import get_ipython
            if get_ipython() is not None and 'IPKernelApp' in get_ipython().config:
                return True
        except:
            pass
        return False

    def _setup_data(self):
        """Handles dataset and dataloader creation"""
        self.modality = self.config['modality']
    
        self.train_dataset, self.val_dataset, self.test_dataset, self.num_classes, self.class_names = create_datasets(
            config=self.config,
            dataset_name=self.config['dataset_name'],
            modality=self.modality
        )
        print(f"Dataset '{self.config['dataset_name']}' loaded: {self.num_classes} classes -> {self.class_names}")
        self.train_loader, self.val_loader = create_dataloaders(
            train_dataset=self.train_dataset,
            val_dataset=self.val_dataset,
            config=self.config['data_loading'],
            batch_size_train=self.config['training']['batch_size'],
            batch_size_val=self.config['training']['val_batch_size']
        )
    
        if self.test_dataset:
            self.test_loader = DataLoader(
                self.test_dataset,
                batch_size=self.config['training'].get('eval_batch_size', self.config['training']['batch_size']),
                shuffle=False,
                num_workers=self.config['data_loading'].get('num_workers', 0),
                pin_memory=self.config['data_loading'].get('pin_memory', True),
                prefetch_factor=self.config['data_loading'].get('prefetch_factor', 2) if self.config['data_loading'].get('num_workers', 0) > 0 else None,
                persistent_workers=True if self.config['data_loading'].get('num_workers', 0) > 0 else False
            )
            print(f"Test loader created with {len(self.test_dataset)} samples.")
        else:
            self.test_loader = None
            print("No test dataset loaded.")

    def setup_model(self):
        """Initialize the segmentation model"""
        print(f"Setting up {self.config['modality']} segmentation model with {self.num_classes} classes.")
        self.model = build_model_finetune(self.config, num_classes=self.num_classes)
        self.model.to(self.device)

    def setup_training(self):
        training_config = self.config['training']
        label_key_for_freq_calc = 'label'

        final_class_weights = None
        weighting_config = training_config['loss'].get('weighting', {})
        weighting_strategy = weighting_config.get('strategy', 'none')

        if weighting_strategy in ['frequency', 'square_root'] and self.train_loader:
            class_frequencies = calculate_class_frequencies(
                self.train_loader,
                self.num_classes,
                label_key_for_freq_calc,
                self.device
            )
            if class_frequencies is not None:
                final_class_weights = calculate_class_weights_from_frequencies(
                    class_frequencies,
                    weighting_strategy,
                    self.device,
                    num_classes_for_norm=self.num_classes
                )
            else:
                print(f"Warning: Class frequency calculation for strategy '{weighting_strategy}' returned None.")
                weighting_strategy = 'none'
        elif weighting_strategy == 'custom' and 'custom_weights' in weighting_config:
            custom_weights_list = weighting_config['custom_weights']
            if custom_weights_list and len(custom_weights_list) == self.num_classes:
                final_class_weights = torch.tensor(custom_weights_list, dtype=torch.float, device=self.device)
            else:
                print(f"Warning: 'custom_weights' invalid or length mismatch.")
                weighting_strategy = 'none'
        else:
            weighting_strategy = 'none'
            final_class_weights = None

        loss_cfg_dict = training_config['loss']
        self.criterion = SegmentationLoss(
            loss_type=loss_cfg_dict.get('type', 'ce_dice'),
            num_classes=self.num_classes,
            ce_weight=loss_cfg_dict.get('ce_weight', 1.0),
            dice_weight=loss_cfg_dict.get('dice_weight', 1.0),
            class_weights=final_class_weights
        ).to(self.device)

        if final_class_weights is not None:
            print(f"Loss: {loss_cfg_dict.get('type', 'ce_dice')} with weighting: {weighting_strategy}")
        else:
            print(f"Loss: {loss_cfg_dict.get('type', 'ce_dice')} (no weights)")

        self.setup_optimizer()

        total_epochs = training_config['epochs']
        warmup_epochs = training_config.get('warmup_epochs', 0)
        scheduler_params = training_config.get('lr_scheduler', {}).get('params', {})

        from torch.optim.lr_scheduler import PolynomialLR
        self.scheduler = PolynomialLR(
            self.optimizer,
            total_iters=total_epochs - warmup_epochs if warmup_epochs > 0 else total_epochs,
            power=scheduler_params.get('power', 0.9)
        )


        if warmup_epochs > 0 and self.scheduler is not None:
            self.scheduler = GradualWarmupScheduler(
                self.optimizer,
                multiplier=1.0,
                total_epoch=warmup_epochs,
                after_scheduler=self.scheduler
            )

        self.use_amp = training_config.get('amp', True) and torch.cuda.is_available()
        self.scaler = torch.amp.GradScaler('cuda') if self.use_amp else None
        print("Using AMP" if self.use_amp else "AMP disabled")

        self.best_val_miou = 0.0
        self.best_epoch = 0

    def setup_optimizer(self):
        """Setup optimizer using model's predefined parameter groups"""
        training_config = self.config['training']

        if hasattr(self.model, 'param_groups') and self.model.param_groups:
            print("Using model's predefined parameter groups for optimizer")
            for i, group in enumerate(self.model.param_groups):
                param_count = sum(p.numel() for p in group['params'] if p.requires_grad)
                group_name = group.get('name', f'group_{i}')
                print(f"  - {group_name}: {param_count:,} params, LR: {group['lr']:.2e}")

            self.optimizer = torch.optim.AdamW(
                self.model.param_groups,
                weight_decay=training_config.get('weight_decay', 0.01)
            )
        else:
            print("No predefined parameter groups found, using fallback setup")
            lr = training_config['learning_rate']
            head_params, backbone_params = [], []
            for name, param in self.model.named_parameters():
                if not param.requires_grad:
                    continue
                (head_params if 'seg_head' in name else backbone_params).append(param)

            param_groups = []
            if backbone_params:
                backbone_lr = lr * training_config.get('backbone_lr_factor', 0.1)
                param_groups.append({'params': backbone_params, 'lr': backbone_lr})
                print(f"  - Backbone: {sum(p.numel() for p in backbone_params):,} params, LR: {backbone_lr:.2e}")

            if head_params:
                param_groups.append({'params': head_params, 'lr': lr})
                print(f"  - Head: {sum(p.numel() for p in head_params):,} params, LR: {lr:.2e}")

            if param_groups:
                self.optimizer = torch.optim.AdamW(param_groups, weight_decay=training_config.get('weight_decay', 0.01))
            else:
                self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=training_config.get('weight_decay', 0.01))
                print(f"  - All: {sum(p.numel() for p in self.model.parameters()):,} params, LR: {lr:.2e}")

    def train_epoch(self, epoch, batch_pbar=None):
        """Train for one epoch"""
        self.model.train()
        total_loss = 0
        modality = self.config['modality']
        accum_steps = self.config['training'].get('gradient_accumulation_steps', 1)
        
        num_batches = len(self.train_loader)
        
        # Reset batch progress bar if provided
        if batch_pbar is not None:
            batch_pbar.reset(total=num_batches)
            batch_pbar.set_description(f"Epoch {epoch+1} batches")
            iterator = enumerate(self.train_loader)
        else:
            iterator = tqdm(enumerate(self.train_loader), total=num_batches, 
                           desc=f"Epoch {epoch+1}/{self.config['training']['epochs']}", leave=False)

        for i, batch in iterator:
            if modality == 'rgb_hsi':
                inputs = {
                    'rgb': batch['rgb'].to(self.device, non_blocking=True),
                    'hsi': batch['hsi'].to(self.device, non_blocking=True)
                }
            else:
                inputs = batch['image'].to(self.device, non_blocking=True)

            labels = batch['label'].to(self.device, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=self.use_amp):
                outputs = self.model(inputs)
                if outputs.shape[2:] != labels.shape[1:]:
                    outputs = F.interpolate(outputs, size=labels.shape[1:], mode='bilinear', align_corners=False)
                loss = self.criterion(outputs, labels)

            if accum_steps > 1:
                loss = loss / accum_steps

            if self.scaler:
                self.scaler.scale(loss).backward()
            else:
                loss.backward()

            if (i + 1) % accum_steps == 0 or (i + 1) == num_batches:
                if self.config['training'].get('clip_grad_norm', 0) > 0 and self.scaler:
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config['training']['clip_grad_norm'])
                elif self.config['training'].get('clip_grad_norm', 0) > 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config['training']['clip_grad_norm'])

                if self.scaler:
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                else:
                    self.optimizer.step()

                self.optimizer.zero_grad()

            current_loss = loss.item() * accum_steps if accum_steps > 1 else loss.item()
            total_loss += current_loss
            
            # Update progress bar
            if batch_pbar is not None:
                batch_pbar.update(1)
                batch_pbar.set_postfix(loss=f"{current_loss:.4f}", lr=f"{self.optimizer.param_groups[0]['lr']:.2e}")
            elif hasattr(iterator, 'set_postfix'):
                iterator.set_postfix(loss=f"{current_loss:.4f}", lr=f"{self.optimizer.param_groups[0]['lr']:.2e}")

        avg_loss = total_loss / num_batches

        if WANDB_AVAILABLE and self.config.get('wandb', {}).get('enable', False) and self.is_main_process:
            wandb.log({'train/epoch_loss': avg_loss, 'train/lr': self.optimizer.param_groups[0]['lr']}, step=epoch + 1)

        return avg_loss

    def validate_epoch(self, epoch, is_test=False):
        """Validate the model on the validation set or test set"""
        self.model.eval()
        total_loss = 0
        model_modality = self.config['modality']
        num_classes = self.num_classes

        current_loader = self.test_loader if is_test else self.val_loader
        if not current_loader:
            print(f"Warning: {'Test' if is_test else 'Validation'} loader not available.")
            return (0, {}) if not is_test else {}

        log_prefix = "test" if is_test else "val"

        all_preds_list, all_labels_list = [], []
        vis_images_list, vis_labels_list, vis_preds_list = [], [], []

        vis_cfg = self.config.get('visualization', {})
        vis_max_images = vis_cfg.get('max_images', 4)
        vis_log_interval = int(vis_cfg.get('log_interval', 1))
        max_vis_samples = 0 if is_test else (vis_max_images if ((epoch + 1) % max(1, vis_log_interval) == 0) else 0)

        with torch.no_grad():
            for batch_idx, batch in enumerate(current_loader):
                labels = batch['label'].to(self.device, non_blocking=True)

                if model_modality == 'rgb_hsi':
                    images = {
                        'rgb': batch['rgb'].to(self.device, non_blocking=True),
                        'hsi': batch['hsi'].to(self.device, non_blocking=True)
                    }
                elif model_modality in ('rgb', 'hsi'):
                    images = batch['image'].to(self.device, non_blocking=True)
                else:
                    raise ValueError(f"Unsupported model_modality: {model_modality}")

                with torch.amp.autocast('cuda', enabled=self.scaler is not None):
                    logits = self.model(images)
                    if logits.shape[2:] != labels.shape[1:]:
                        logits = F.interpolate(logits, size=labels.shape[1:], mode='bilinear', align_corners=False)
                    loss = self.criterion(logits, labels)
                total_loss += loss.item()

                preds = logits.argmax(dim=1)
                all_preds_list.append(preds.cpu())
                all_labels_list.append(labels.cpu())

                if not is_test and batch_idx < (max_vis_samples // current_loader.batch_size) + 1:
                    if isinstance(images, dict):
                        img_to_vis = images.get('rgb', images.get('hsi'))
                        if img_to_vis is not None:
                            vis_images_list.append(img_to_vis.cpu())
                    else:
                        vis_images_list.append(images.cpu())
                    vis_labels_list.append(labels.cpu())
                    vis_preds_list.append(logits.cpu())

        avg_loss = total_loss / len(current_loader)
        all_preds = torch.cat(all_preds_list).numpy()
        all_labels = torch.cat(all_labels_list).numpy()
        metrics = calculate_metrics(all_preds, all_labels, num_classes)

        # WandB logging
        if not is_test and self.is_main_process and WANDB_AVAILABLE and self.config.get('wandb', {}).get('enable', False):
            labels_cat = torch.cat(all_labels_list)
            preds_cat = torch.cat(all_preds_list)
        
            if vis_images_list:
                vis_images_cat = torch.cat(vis_images_list)[:max_vis_samples]
            else:
                H, W = labels_cat.shape[1], labels_cat.shape[2]
                n = min(labels_cat.shape[0], max_vis_samples)
                vis_images_cat = torch.zeros((n, 3, H, W), dtype=torch.float)
        
            class_names = self.class_names if self.class_names is not None else [f"Class {i}" for i in range(self.num_classes)]
            modality_for_vis = 'rgb' if model_modality == 'rgb_hsi' else model_modality
        
            log_dict = {}
            if 'miou' in metrics:
                log_dict[f'{log_prefix}/miou'] = metrics['miou']
            if 'class_iou' in metrics:
                for i in range(num_classes):
                    cname = class_names[i] if i < len(class_names) else f"Class_{i}"
                    log_dict[f'{log_prefix}/iou/{cname}'] = metrics['class_iou'][i]
        
            wandb_imgs_log = {}
            try:
                from pyfunctions.wandb_image_visualization import prepare_wandb_logs_SpectralWaste
                iou_metrics, images_list = prepare_wandb_logs_SpectralWaste(
                    vis_images_cat, labels_cat, preds_cat,
                    num_classes=self.num_classes,
                    modality=modality_for_vis,
                    class_names=class_names,
                    max_images=max_vis_samples,
                    split='val'
                )
                if images_list:
                    wandb_imgs_log = {f'{log_prefix}/samples': images_list}
            except Exception as e:
                pass  # Silently skip visualization errors
        
            if log_dict:
                wandb.log({**log_dict, **wandb_imgs_log}, step=epoch + 1)

        if is_test:
            return metrics
        return avg_loss, metrics

    def save_checkpoint(self, epoch, is_best=False):
        """Save a checkpoint of the model"""
        if not self.is_main_process:
            return

        checkpoint_name = "best.pth" if is_best else f"epoch_{epoch}.pth"
        checkpoint_path = self.checkpoint_dir / checkpoint_name

        def _uniformize_fusion_state_dict(sd):
            from collections import OrderedDict
            out = OrderedDict()
            backbone_roots = ('backbone.', 'patch_embed.', 'layers.', 'norm.', 'pos_drop.', 'stages.', 'hf_model.')
            for k, v in sd.items():
                if k.startswith("rgb_model.") and not k.startswith("rgb_model.backbone."):
                    suffix = k[len("rgb_model."):]
                    out[f"rgb_model.backbone.{suffix}" if suffix.startswith(backbone_roots) else k] = v
                elif k.startswith("hsi_model.") and not k.startswith("hsi_model.backbone."):
                    suffix = k[len("hsi_model."):]
                    out[f"hsi_model.backbone.{suffix}" if suffix.startswith(backbone_roots) else k] = v
                else:
                    out[k] = v
            return out

        model_state_dict = self.model.state_dict()
        model_state_dict = _uniformize_fusion_state_dict(model_state_dict)

        checkpoint = {
            'epoch': epoch,
            'model': model_state_dict,
            'optimizer': self.optimizer.state_dict(),
            'scheduler': self.scheduler.state_dict() if self.scheduler else None,
            'best_miou': self.best_val_miou
        }

        torch.save(checkpoint, checkpoint_path)

        if is_best or epoch == self.config['training']['epochs']:
            model_name_prefix = self.config['training'].get('model_name', self.config.get('name', 'model'))
            final_model_name = f"{model_name_prefix}_best.pth" if is_best else f"{model_name_prefix}_epoch_{epoch}.pth"
            final_model_path = self.model_dir / final_model_name
            torch.save(model_state_dict, final_model_path)

    def load_checkpoint(self, checkpoint_path):
        """Load a checkpoint to resume training"""
        resolved_path = Path(checkpoint_path)
        if not resolved_path.exists():
            resolved_path = self.checkpoint_dir / checkpoint_path
            if not resolved_path.exists():
                raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

        print(f"Loading checkpoint: {resolved_path}")
        checkpoint = torch.load(resolved_path, map_location=self.device, weights_only=False)

        model_state_dict = checkpoint['model']
        if any(key.startswith('_orig_mod.') for key in model_state_dict.keys()):
            print("Removing _orig_mod. prefixes...")
            model_state_dict = {k[10:] if k.startswith('_orig_mod.') else k: v for k, v in model_state_dict.items()}

        self.model.load_state_dict(model_state_dict)
        self.optimizer.load_state_dict(checkpoint['optimizer'])

        if checkpoint.get('scheduler') and self.scheduler:
            try:
                self.scheduler.load_state_dict(checkpoint['scheduler'])
            except Exception as e:
                print(f"Warning: Could not load scheduler state: {e}")

        self.best_val_miou = checkpoint.get('best_miou', 0.0)
        start_epoch = checkpoint.get('epoch', 0)
        self.best_epoch = start_epoch
        print(f"Resumed from epoch {start_epoch} with best mIoU {self.best_val_miou:.4f}")
        return start_epoch

    def train(self):
        """Main training loop with clean Jupyter output"""
        start_epoch = 0
        total_epochs = self.config['training']['epochs']
        
        # Initial info
        print(f"{'='*60}")
        print(f"Starting training for {total_epochs} epochs")
        print(f"   Dataset: {self.config['dataset_name']}")
        print(f"   Model: {self.config.get('model', {}).get('name', 'unknown')}")
        print(f"   Modality: {self.config['modality']}")
        print(f"   Classes: {self.num_classes}")
        print(f"{'='*60}")
        
        # Create epoch progress bar
        epoch_pbar = tqdm(
            range(start_epoch, total_epochs), 
            desc="Training", 
            unit="epoch",
            leave=True,
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}'
        )
        
        # Create batch progress bar (reused each epoch)
        batch_pbar = tqdm(
            total=len(self.train_loader), 
            desc="Batches", 
            leave=False,
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} {postfix}'
        )
        
        for epoch in epoch_pbar:
            # Train one epoch
            train_loss = self.train_epoch(epoch, batch_pbar=batch_pbar)
            
            # Step scheduler
            if self.scheduler:
                self.scheduler.step()

            # Validate
            val_loss, val_metrics = self.validate_epoch(epoch)
            current_miou = val_metrics['miou']
            
            # Update epoch progress bar
            epoch_pbar.set_postfix({
                'loss': f'{train_loss:.3f}',
                'val_loss': f'{val_loss:.3f}',
                'mIoU': f'{current_miou:.4f}',
                'best': f'{self.best_val_miou:.4f}'
            })
            
            if self.is_main_process:
                is_best = current_miou > self.best_val_miou
                
                if is_best:
                    improvement = current_miou - self.best_val_miou
                    self.best_val_miou = current_miou
                    self.best_epoch = epoch + 1
                    self.save_checkpoint(epoch + 1, is_best=True)
                    
                    tqdm.write(
                        f"NEW BEST @ Epoch {epoch+1:3d}: "
                        f"mIoU={current_miou:.4f} (+{improvement:.4f}) | "
                        f"Train Loss={train_loss:.4f} | "
                        f"Val Loss={val_loss:.4f}"
                    )

                # Save periodic checkpoints
                save_interval = self.config['training'].get('save_checkpoint_interval', 0)
                if save_interval > 0 and (epoch + 1) % save_interval == 0:
                    self.save_checkpoint(epoch + 1)
        
        # Close progress bars
        batch_pbar.close()
        epoch_pbar.close()
        
        # Final summary
        if self.is_main_process:
            print(f"\n{'='*60}")
            print(f"Training Complete!")
            print(f"   Best mIoU: {self.best_val_miou:.4f} (Epoch {self.best_epoch})")
            print(f"{'='*60}")

            # Test evaluation
            if self.test_loader is not None:
                best_checkpoint_path = self.checkpoint_dir / "best.pth"
                if best_checkpoint_path.exists():
                    print(f"\nRunning final test evaluation...")
                    self.load_checkpoint(str(best_checkpoint_path))
                    test_metrics = self.validate_epoch(epoch=total_epochs - 1, is_test=True)
                    
                    class_names = self.class_names if self.class_names is not None else [f"Class {i}" for i in range(self.num_classes)]
                    
                    print(f"\n{'='*60}")
                    print(f"Test Set Results:")
                    print(f"{'-'*60}")
                    for metric_name, metric_val in test_metrics.items():
                        if metric_name == 'class_iou':
                            continue  # printed separately below
                        if isinstance(metric_val, float):
                            print(f"   {metric_name:20s}: {metric_val:.4f}")
                        if WANDB_AVAILABLE and self.config.get('wandb', {}).get('enable', False):
                            wandb.log({f"final_test/{metric_name}": metric_val})
                    
                    # Per-class IoU
                    if 'class_iou' in test_metrics:
                        print(f"{'-'*60}")
                        print(f"   Per-Class IoU:")
                        for i, iou_val in enumerate(test_metrics['class_iou']):
                            cname = class_names[i] if i < len(class_names) else f"Class_{i}"
                            print(f"     {cname:20s}: {iou_val:.4f}")
                            if WANDB_AVAILABLE and self.config.get('wandb', {}).get('enable', False):
                                wandb.log({f"final_test/iou/{cname}": iou_val})
                    
                    print(f"{'='*60}")
                else:
                    print("Warning: best.pth not found, skipping final test evaluation.")
            else:
                print("No test loader configured.")

        if WANDB_AVAILABLE and self.config.get('wandb', {}).get('enable', False) and self.is_main_process:
            wandb.finish()

### Run

In [ ]:
# Load your config YAML
config_path = "/home/jon86439/BCAF_2026/ModelConfigs/Fusion/BCAF_RGB_256_HSI_5.yaml"

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

for key in ['rgb_model_config_path', 'rgb_checkpoint_path',
            'hsi_model_config_path', 'hsi_checkpoint_path']:
    if key in config and not Path(config[key]).is_absolute():
        config[key] = str(project_root / config[key])

# Set random seed if specified
seed = config.get('training', {}).get('random_seed')
if seed is not None:
    print(f"Setting random seed to: {seed}")
    torch.manual_seed(seed)
    np.random.seed(seed)
else:
    print("No random_seed specified in training config.")

# Initialize trainer and start training
trainer = SegmentationTrainer(config)
trainer.train()

print("Done!")